<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part3_transformers/11_text_in_images_and_multimodal_posts/notebooks/11_text_in_images_and_multimodal_posts_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 11: Text in Images and Multimodal Posts

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Much of what social scientists study is neither pure text nor pure image. A meme says one thing in
its caption and another in its picture. A cereal box makes health claims in print next to a cartoon
mascot. A post pairs a photo with a few words that change how it reads. This notebook is about text
inside images, and about content whose meaning lives in the combination.

In this notebook you will:

- read the text on packaging with classic OCR (Tesseract, from module 03) and with the VLM from
  module 10, and measure how much of the product name each one finds
- have the VLM code health claims on the packaging and compare them with the labels in the Open
  Food Facts database
- make a small set of memes in which the same photo with a different caption flips the meaning, and
  test which models notice
- compare text only, image only and both for predicting a product's Nutri-Score

**Data.** Open Food Facts cereal packaging with its product data (photos CC BY-SA, database ODbL),
and protest and King's Day photos from Wikimedia Commons for the memes.

**Models.** Tesseract, CLIP from module 09 and `Qwen/Qwen3-VL-2B-Instruct` from module 10.

**Runtime.** You need a GPU: *Runtime → Change runtime type → T4 GPU*. Reading and coding 120
packages with the VLM takes roughly twenty minutes on a T4.

## 0. Setup

Tesseract and its Dutch language data come from Ubuntu's package manager, as in module 03.

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-nld > /dev/null
!pip install -q pytesseract "transformers>=4.57"

In [ ]:
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import requests
import torch
import pytesseract
from PIL import Image, ImageDraw, ImageFont
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score

SEED = 42
np.random.seed(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

print("Running on:", DEVICE)
ON_GPU = DEVICE == "cuda"

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images (file paths or PIL images) in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(path if isinstance(path, Image.Image) else load_rgb(path, 300))
        ax.set_title(str(label)[:32], fontsize=7)
    plt.tight_layout()
    plt.show()

packaging = course_data.get_packaging_corpus()
protests = course_data.get_protest_corpus()
koningsdag = course_data.get_koningsdag_corpus()

## 1. Getting the text out: OCR versus a VLM

Tesseract, which you used in module 03, finds lines of text and recognizes them character by
character. It works well on clean, horizontal, dark-on-light print and struggles with stylized
logos, curved text, low contrast and small type. A VLM reads differently: it sees the whole image
and *writes* the text it believes is there, so it copes better with odd fonts and layouts, and it
can also produce text that isn't there.

We need a yardstick. The Open Food Facts database has each product's **name**, typed in by
contributors, and that name is almost always printed on the front. So we measure **name recall**:
the share of the words in the product name that appear in the extracted text. It doesn't measure
everything (it ignores the rest of the text, and it can't catch invented text), but it is a fair,
automatic comparison.

In [ ]:
N_READ = 120 if ON_GPU else 15
sample = packaging.dropna(subset=["product_name"]).sample(min(N_READ, packaging["product_name"].notna().sum()),
                                                          random_state=SEED).reset_index(drop=True)

def normalize(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode().lower()
    return re.findall(r"[a-z0-9]{3,}", text)

def name_recall(name, extracted):
    words = set(normalize(name))
    return np.nan if not words else len(words & set(normalize(extracted))) / len(words)

READ_PROMPT = "Transcribe all the text printed on the front of this package, line by line. Write only the text, nothing else."

start = time.time()
sample["tesseract"] = [pytesseract.image_to_string(load_rgb(p, 1200), lang="eng+nld") for p in sample["local_path"]]
print(f"Tesseract: {time.time() - start:.0f}s")
start = time.time()
sample["vlm"] = [course_torch.ask_vlm(p, READ_PROMPT, max_new_tokens=150, max_side=1024) for p in sample["local_path"]]
print(f"VLM: {time.time() - start:.0f}s")

for reader in ["tesseract", "vlm"]:
    sample[f"recall_{reader}"] = [name_recall(n, t) for n, t in zip(sample["product_name"], sample[reader])]
sample[["recall_tesseract", "recall_vlm"]].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.hist([sample["recall_tesseract"], sample["recall_vlm"]], bins=np.linspace(0, 1, 6), label=["Tesseract", "VLM"])
ax.set_xlabel("share of product-name words found")
ax.set_ylabel("packages")
ax.legend()
plt.show()

row = sample.iloc[0]
show_grid([row["local_path"]], [row["product_name"]], n_cols=1, size=3.5)
print("TESSERACT:\n", row["tesseract"].strip()[:300], "\n\nVLM:\n", row["vlm"][:300])

> **✏️ Exercise 1**
>
> Find the packages where the two readers differ most: where the VLM found the whole name and
> Tesseract found none of it, and the other way around. Show a few of each with both transcriptions.
> What kinds of text does each reader miss? Then look for signs of **invented text**: words in the
> VLM's transcription that Tesseract didn't find *and* that you can't see on the package.

In [ ]:
# Your code here

## 2. Coding health claims, and checking them against the database

Packaging makes **health claims** ("whole grain", "high in fiber", "no added sugar", "organic"),
and those claims are a classic topic in health communication. Open Food Facts records many of them
as **labels**, entered by contributors. Those labels are an imperfect benchmark: some are missing,
and some describe the product rather than what the front of the box says. But they let us check
the VLM's coding without hand-coding anything.

In [ ]:
CLAIMS_PROMPT = """Look at the front of this breakfast cereal package. Does it show any of these claims, in words or in a logo?
- whole_grain: whole grain, wholegrain or volkoren
- fiber: high in fiber, source of fiber, rich in fiber or vezelrijk
- sugar: no added sugar, less sugar, low in sugar or minder suiker
- organic: organic, bio or an organic logo
Answer with JSON only, in exactly this format (1 = shown, 0 = not shown):
{"whole_grain": 0, "fiber": 0, "sugar": 0, "organic": 0}"""

CLAIMS = ["whole_grain", "fiber", "sugar", "organic"]
LABEL_PATTERNS = {"whole_grain": r"whole-?grain|wholemeal|volkoren", "fiber": r"fib(re|er)", "sugar": r"sugar|suiker",
                  "organic": r"organic|\bbio\b|biolog"}

def labels_to_claims(tags):
    tags = " ".join(tags) if isinstance(tags, (list, np.ndarray)) else str(tags)
    return {c: int(bool(re.search(p, tags))) for c, p in LABEL_PATTERNS.items()}

database = pd.DataFrame([labels_to_claims(t) for t in sample["labels_tags"]])
start = time.time()
answers = [course_torch.parse_json(course_torch.ask_vlm(p, CLAIMS_PROMPT, max_new_tokens=40, max_side=1024)) or {}
           for p in sample["local_path"]]
vlm_claims = pd.DataFrame([{c: int(a.get(c, 0) in (1, "1", True)) for c in CLAIMS} for a in answers])
print(f"{len(sample)} packages in {time.time() - start:.0f}s; answers without valid JSON: {sum(not a for a in answers)}")

pd.DataFrame({"share in database labels": database.mean(), "share coded by VLM": vlm_claims.mean(),
              "agreement": (database == vlm_claims).mean()}).round(2)

> **✏️ Exercise 2**
>
> For each claim, make a 2 × 2 table of database label versus VLM code. Then look at the packages
> where the VLM says a claim is shown but the database has no such label, and the reverse. Who is
> right in those cases: the VLM, the database, or neither?

In [ ]:
# Your code here

## 3. Memes: the same picture, a different meaning

In a meme the text and the picture comment on each other, and the meaning is often in the
combination. To study that cleanly, we make our own small set: six protest photos, each with two
captions, one supportive of the protest and one mocking it. The picture is identical within each
pair, so **only the caption** can tell the two versions apart. That lets us test what each model
actually uses.

The photos come from Wikimedia Commons under free licenses (mostly CC BY and CC BY-SA). Adding a
caption is an adaptation, which those licenses allow, but if you ever *share* adapted images you
have to credit the author and, for CC BY-SA, use the same license. Here they stay in your session.

In [ ]:
FONT_PATH = Path(matplotlib.get_data_path()) / "fonts/ttf/DejaVuSans-Bold.ttf"

def make_meme(path, top, bottom="", width=500):
    """Put meme-style captions (white text, black outline) on a photo."""
    img = load_rgb(path, 1000)
    img = img.resize((width, int(img.height * width / img.width)))
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(str(FONT_PATH), size=max(18, width // 18))
    for text, y in [(top, 8), (bottom, None)]:
        if not text:
            continue
        box = draw.multiline_textbbox((0, 0), text, font=font, align="center")
        x = (img.width - (box[2] - box[0])) / 2
        y = y if y is not None else img.height - (box[3] - box[1]) - 14
        draw.multiline_text((x, y), text, font=font, fill="white", stroke_width=3, stroke_fill="black", align="center")
    return img

CAPTION_PAIRS = [
    ("THOUSANDS STOOD UP\nFOR WHAT IS RIGHT", "SKIPPED WORK TO\nHOLD A CARDBOARD SIGN"),
    ("THIS IS WHAT\nDEMOCRACY LOOKS LIKE", "THIS IS WHAT\nBOREDOM LOOKS LIKE"),
    ("PROUD OF EVERYONE\nWHO SHOWED UP", "NOBODY CARES\nGO HOME"),
    ("CHANGE STARTS\nIN THE STREETS", "TRAFFIC STARTS\nIN THE STREETS"),
    ("TOGETHER WE\nWILL BE HEARD", "SHOUTING AT\nBUILDINGS AGAIN"),
    ("HISTORY IS MADE\nBY PEOPLE LIKE THESE", "SAME PEOPLE\nEVERY WEEKEND"),
]
photos = protests.sample(len(CAPTION_PAIRS), random_state=SEED)["local_path"].tolist()
memes = []
for k, (path, (support, mock)) in enumerate(zip(photos, CAPTION_PAIRS)):
    memes.append({"pair": k, "photo": path, "caption": support, "stance": "supportive", "image": make_meme(path, support)})
    memes.append({"pair": k, "photo": path, "caption": mock, "stance": "mocking", "image": make_meme(path, mock)})
memes = pd.DataFrame(memes)
show_grid(memes["image"], memes["stance"], n_cols=6, size=2.2)

Now four ways to classify each meme's stance:

1. **Caption only:** the VLM reads just the caption text, as a text classifier would after OCR.
2. **Photo only:** the VLM sees the photo without a caption. By design it can't do better than
   chance: both versions of a pair share the same photo, so whatever it says, it gets one of the
   two wrong.
3. **CLIP on the meme:** similarity of the captioned image to "a meme supporting a protest" versus
   "a meme mocking a protest". CLIP has learned to read some text in images, a side effect of
   training on web images that contain text.
4. **VLM on the meme:** the VLM sees the captioned image.

In [ ]:
STANCE_QUESTION = ("Is this meant to support the protest or to mock it? "
                   "Answer with one word: positive (supportive) or negative (mocking).")

def to_stance(answer):
    return "supportive" if "positive" in answer.lower() or "support" in answer.lower() else "mocking"

memes["caption only"] = [to_stance(course_torch.ask_vlm([], f'Caption of a photo of a protest: "{c}". {STANCE_QUESTION}', max_new_tokens=5))
                         for c in memes["caption"].str.replace("\n", " ")]
memes["photo only"] = [to_stance(course_torch.ask_vlm(p, f"This photo shows a protest. {STANCE_QUESTION}", max_new_tokens=5))
                       for p in memes["photo"]]
memes["VLM on meme"] = [to_stance(course_torch.ask_vlm(img, f"This is a meme about a protest. {STANCE_QUESTION}", max_new_tokens=5))
                        for img in memes["image"]]

meme_emb = course_torch.clip_images(list(memes["image"]))
poles = course_torch.clip_texts(["a meme supporting a protest", "a meme mocking a protest"])
memes["CLIP on meme"] = np.where((meme_emb @ poles.T).argmax(axis=1) == 0, "supportive", "mocking")

METHODS = ["caption only", "photo only", "CLIP on meme", "VLM on meme"]
pd.Series({m: (memes[m] == memes["stance"]).mean() for m in METHODS}, name="accuracy on 12 memes").round(2)

With 12 memes these numbers are rough, but the pattern is usually clear: the photo alone sits at
50% by construction, the caption alone does well, and the methods that see the meme do as well as
they can *read* the caption. The point of the design is that it tells you *why* a model gets a
meme right. Real memes are harder, because there the photo often matters too.

> **✏️ Exercise 3**
>
> Now turn the design around: keep the caption fixed and let the **photo** change the meaning. Put the
> caption `"WHAT A LOVELY DAY\nIN THE CITY"` on six King's Day photos (where it is sincere) and on six
> protest photos (where it reads as sarcastic). Ask the VLM whether each caption is meant sincerely or
> sarcastically, and compute the accuracy for each group. Does the model use the picture to read the
> caption?

In [ ]:
# Your code here

## 4. What does each modality add? Text only, image only, both

A standard way to measure what each part of a multimodal post contributes is an **ablation**: train
the same kind of classifier on the text alone, on the image alone, and on both, and compare. Our
outcome is whether a product has a favorable Nutri-Score (A or B) or an unfavorable one (D or E),
and the question is how much of that can be read off the front of the package.

The text is what Tesseract read in section 1; the image is the CLIP embedding. With about a hundred
products we use 5-fold cross-validation instead of a single split.

In [ ]:
data = sample[sample["nutriscore_grade"].isin(list("abde"))].reset_index(drop=True)
y = data["nutriscore_grade"].isin(["a", "b"]).astype(int).to_numpy()
print(f"{len(data)} products: {y.sum()} with A or B, {len(y) - y.sum()} with D or E")

text_features = TfidfVectorizer(min_df=2, sublinear_tf=True).fit_transform(data["tesseract"].fillna("")).toarray()
image_features = course_torch.clip_images(data["local_path"])

def cv_score(X, y, seed=SEED):
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    model = LogisticRegression(class_weight="balanced", max_iter=3000, C=1.0)
    return balanced_accuracy_score(y, cross_val_predict(model, X, y, cv=folds))

ablation = pd.Series({
    "text only (Tesseract)": cv_score(text_features, y),
    "image only (CLIP)": cv_score(image_features, y),
    "both": cv_score(np.hstack([text_features, image_features]), y),
}, name="balanced accuracy (5-fold CV)").round(2)
ablation

> **✏️ Exercise 4**
>
> Two questions about this ablation. (a) Use the VLM's transcriptions (`data["vlm"]`) as the text
> instead of Tesseract's. Does better reading give a better text-only model? (b) Repeat the whole
> comparison with five different random fold splits (seeds 0 to 4). How much do the scores move, and
> is the ranking of the three versions stable?

In [ ]:
# Your code here

> **✏️ Exercise 5**
>
> You want to study how political parties combine images and text in their Instagram posts. Sketch a
> design that shows what each modality contributes to a post's meaning (or to its engagement). Which
> comparisons would you run, and what would you need to code by hand? If you plan to use an existing
> meme or post dataset, what should you check about its access conditions?

*Your answer here*

In [ ]:
course_data.keep_my_files()